In [1]:
import pandas as pd
from src.pipelines.BERT_pipeline import BERTPipeline
import logging
import torch
import os

In [2]:
df = pd.read_csv("data/aes_dataset_5k_clean.csv")
df = df[df['dataset'] == 'analisis_essay'][['reference_answer', 'answer', 'score', 'normalized_score', 'dataset', 'dataset_num']]
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 2162 entries, 0 to 2161
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   reference_answer  2162 non-null   object 
 1   answer            2162 non-null   object 
 2   score             2162 non-null   float64
 3   normalized_score  2162 non-null   float64
 4   dataset           2162 non-null   object 
 5   dataset_num       2162 non-null   object 
dtypes: float64(2), object(4)
memory usage: 118.2+ KB
None


,reference_answer,answer,score,normalized_score,dataset,dataset_num
0,Fungsi karbohidrat adalah sebagai pemasok ener...,"sumber tenaga, pemanis alami, menjaga sistem i...",27.0,0.27,analisis_essay,analisis_essay-1
1,Fungsi karbohidrat adalah sebagai pemasok ener...,"sebagai sumber energi, pemanis alami, menjaga ...",21.0,0.21,analisis_essay,analisis_essay-1
2,Fungsi karbohidrat adalah sebagai pemasok ener...,1. Sebagai energi. 2. Sebagai memperlancaar pe...,42.0,0.42,analisis_essay,analisis_essay-1
3,Fungsi karbohidrat adalah sebagai pemasok ener...,"untuk membuat kenyang, agar tidak lapar, agar ...",18.0,0.18,analisis_essay,analisis_essay-1
4,Fungsi karbohidrat adalah sebagai pemasok ener...,Karbohidrat mempunyai peran penting untuk pros...,82.0,0.82,analisis_essay,analisis_essay-1


In [3]:
# Check if the first file exists
df_result = None
if os.path.exists("experiments/results/results_bert_mlr_after.csv"):
    df_result = pd.read_csv("experiments/results/results_bert_mlr_after.csv")
    print(df_result['config_id'].iloc[-1])
else:
    print("File 'results_bert_mlr_after.csv' does not exist.")

File 'results_bert_mlr_after.csv' does not exist.


In [4]:
experiments = [(8, 2e-5, 1e-3), (16, 2e-5, 2e-3)]
dropouts = [0.3, 0.5]
idx = (df_result['config_id'].iloc[-1] + 1) if df_result is not None and not df_result.empty else 0  # index untuk setiap kombinasi
ROOT_DIR = os.getcwd()

In [5]:
for experiment in experiments:
    for dropout in dropouts:
        results = []
        results_epoch = []
        df_result1 = None
        # Check if the second file exists
        if os.path.exists("experiments/results/results_epoch_bert_mlr_after.csv"):
            df_result1 = pd.read_csv("experiments/results/results_epoch_bert_mlr_after.csv")
            print(max(df_result1['valid_pearson']))
        else:
            print("File 'results_epoch_bert_mlr_after.csv' does not exist.")

        # set up hyperparamter
        config = {
            "df": df,
            "model_name": "indobenchmark/indobert-lite-base-p2",
            "batch_size": experiment[0],
            "learning_rate_backbone": experiment[1],
            "learning_rate_head": experiment[2],
            "epochs": 100,
            "config_id": idx,
            "best_valid_pearson": max(df_result1['valid_pearson']) if df_result1 is not None and not df_result1.empty else float("-inf"),
            "warmup_ratio": 0.0,
            "use_reference": True,
            "dropout": dropout,
        }

        logging.info(
            f"Running configuration: config_id={idx}, model_name={config['model_name']}"
            f", batch_size={experiment[0]}, epochs={100}, learning_rate_backbone={experiment[1]}, learning_rate_head={experiment[2]}"
        )
        
        print(
            f"\nRunning configuration: config_id={idx}, model_name={config['model_name']}"
            f", batch_size={experiment[0]}, epochs={100}, learning_rate_backbone={experiment[1]}, learning_rate_head={experiment[2]}"
        )
        
        try:
            pipeline = BERTPipeline(config, results, results_epoch)
            pipeline.training()

            # Save results
            # Dapatkan root project
            results_path = os.path.join(ROOT_DIR, "experiments/results/results_bert_mlr_after.csv")
            results_epoch_path = os.path.join(ROOT_DIR, "experiments/results/results_epoch_bert_mlr_after.csv")
            BERTPipeline.save_csv(results, results_path)
            BERTPipeline.save_csv(results_epoch, results_epoch_path)
        except Exception as e:
            logging.error(f"Error in config_id={idx}: {str(e)}")
            print(f"Error in config_id={idx}: {str(e)}")
            torch.cuda.empty_cache()
        finally:
            # Clear GPU memory after every configuration
            del pipeline.model
            del pipeline.tokenizer
            del pipeline.optimizer
            torch.cuda.empty_cache()

        idx += 1

File 'results_epoch_bert_mlr_after.csv' does not exist.

Running configuration: config_id=0, model_name=indobenchmark/indobert-lite-base-p2, batch_size=8, epochs=100, learning_rate_backbone=2e-05, learning_rate_head=0.001


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'AlbertTokenizer'. 
The class this function is called from is 'BertTokenizer'.


run split dataset...
run split dataset...
create dataset run...
create dataloader run...
====== Training Epoch 1/100 ======


c:\Users\User\Documents\Code\env\lib\site-packages\transformers\models\albert\modeling_albert.py:404: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attention_output = torch.nn.functional.scaled_dot_product_attention(
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list wi

Epoch 1/100 - Avg training loss: 0.0526, MAE: 0.1673, RMSE: 0.2293, Pearson Corr: 0.6843
Avg validation loss: 0.0174, MAE: 0.1059, RMSE: 0.1318, Pearson Corr: 0.9018
Validation loss decreased (inf --> 0.017411). Saving model ...
Model saved to experiments\models\indobenchmark/indobert-lite-base-p2_best_model.pt
====== Training Epoch 2/100 ======


...

Epoch 20/100 - Avg training loss: 0.0031, MAE: 0.0436, RMSE: 0.0558, Pearson Corr: 0.9788
Avg validation loss: 0.0087, MAE: 0.07159, RMSE: 0.09233, Pearson Corr: 0.9399
EarlyStopping counter: 9 out of 10
====== Training Epoch 21/100 ======


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


Epoch 21/100 - Avg training loss: 0.0034, MAE: 0.04581, RMSE: 0.05824, Pearson Corr: 0.9769
Avg validation loss: 0.0076, MAE: 0.06759, RMSE: 0.08651, Pearson Corr: 0.9411
EarlyStopping counter: 10 out of 10
Early stopping triggered


c:\Users\User\Documents\Code\aes\src\pipelines\BERT_pipeline.py:119: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('experiments/models/checkpoint.pt'

Avg testing loss: 0.0074, MAE: 0.0656, RMSE: 0.08601, Pearson Corr: 0.9435


In [6]:
# batch_sizes = [4, 8, 16]
# learning_rates_backbone = [1e-5, 2e-5]
# learning_rates_head = [1e-3, 2e-3]
# use_references = [False]
# warm_ups = [0.0, 0.3]
# idx = (df_result['config_id'].iloc[-1] + 1) if df_result is not None and not df_result.empty else 0  # index untuk setiap kombinasi
# ROOT_DIR = os.getcwd()

In [7]:
# for use_ref in use_references:
#     for warm_up in warm_ups:
#         for batch_size in batch_sizes:
#             for lr_backbone in learning_rates_backbone:
#                 for lr_head in learning_rates_head:
#                     results = []
#                     results_epoch = []
#                     df_result1 = None
#                     # Check if the second file exists
#                     if os.path.exists("experiments/results/results_epoch_bert_1.csv"):
#                         df_result1 = pd.read_csv("experiments/results/results_epoch_bert_1.csv")
#                         print(max(df_result1['valid_pearson']))
#                     else:
#                         print("File 'results_epoch_bert_1.csv' does not exist.")

#                     # set up hyperparamter
#                     config = {
#                         "df": df,
#                         "model_name": "indobenchmark/indobert-lite-base-p2",
#                         "batch_size": batch_size,
#                         "learning_rate_backbone": lr_backbone,
#                         "learning_rate_head": lr_head,
#                         "epochs": 100,
#                         "config_id": idx,
#                         "best_valid_pearson": max(df_result1['valid_pearson']) if df_result1 is not None and not df_result1.empty else float("-inf"),
#                         "warmup_ratio": warm_up,
#                         "use_reference": use_ref,
#                     }

#                     logging.info(
#                         f"Running configuration: config_id={idx}, model_name={config['model_name']}"
#                         f", batch_size={batch_size}, epochs={100}, learning_rate_backbone={lr_backbone}, learning_rate_head={lr_head}"
#                     )
                    
#                     print(
#                         f"\nRunning configuration: config_id={idx}, model_name={config['model_name']}"
#                         f", batch_size={batch_size}, epochs={100}, learning_rate_backbone={lr_backbone}, learning_rate_head={lr_head}"
#                     )
                    
#                     try:
#                         pipeline = BERTPipeline(config, results, results_epoch)
#                         pipeline.training()

#                         # Save results
#                         # Dapatkan root project
#                         results_path = os.path.join(ROOT_DIR, "experiments/results/results_bert_1.csv")
#                         results_epoch_path = os.path.join(ROOT_DIR, "experiments/results/results_epoch_bert_1.csv")
#                         BERTPipeline.save_csv(results, results_path)
#                         BERTPipeline.save_csv(results_epoch, results_epoch_path)
#                     except Exception as e:
#                         logging.error(f"Error in config_id={idx}: {str(e)}")
#                         print(f"Error in config_id={idx}: {str(e)}")
#                         torch.cuda.empty_cache()
#                     finally:
#                         # Clear GPU memory after every configuration
#                         del pipeline.model
#                         del pipeline.tokenizer
#                         del pipeline.optimizer
#                         torch.cuda.empty_cache()

#                     idx += 1